In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, kruskal, mannwhitneyu

from src.evaluate import compute_correlations, stratified_analysis

sns.set_theme(style='whitegrid')

In [ ]:
rows = [json.loads(l) for l in open('../outputs/predictions/adaptive_rag_results.jsonl')]
df   = pd.DataFrame(rows)

qpp_scores   = df['qpp_score'].values.astype(float)
rouge_scores = df['rouge_l'].values.astype(float)
bert_scores  = df['bert_f1'].values.astype(float)
mrr_scores   = df['mrr'].values.astype(float) if 'mrr' in df.columns else np.zeros(len(df))

print(f'Loaded {len(df)} results')
print(df[['qpp_score', 'rouge_l', 'bert_f1']].describe().round(4))

In [ ]:
pr_rouge, p1 = pearsonr(qpp_scores, rouge_scores)
sr_rouge, p2 = spearmanr(qpp_scores, rouge_scores)
pr_bert,  p3 = pearsonr(qpp_scores, bert_scores)
sr_bert,  p4 = spearmanr(qpp_scores, bert_scores)

df_corr = pd.DataFrame({
    'Metric':        ['ROUGE-L', 'BERTScore-F1'],
    'Pearson':       [round(pr_rouge, 4), round(pr_bert, 4)],
    'Pearson p':     [round(p1, 4),       round(p3, 4)],
    'Spearman':      [round(sr_rouge, 4), round(sr_bert, 4)],
    'Spearman p':    [round(p2, 4),       round(p4, 4)],
}).set_index('Metric')

df_corr.to_csv('../outputs/predictions/generation_correlations.csv')
print(df_corr)

In [ ]:
bins = stratified_analysis(qpp_scores, mrr_scores, rouge_scores, bert_scores)

df_bins = pd.DataFrame(bins).T[['n', 'mrr', 'rouge_l', 'bert_f1']]
df_bins.columns = ['N', 'MRR@10', 'ROUGE-L', 'BERTScore-F1']
df_bins.to_csv('../outputs/predictions/stratified_bins.csv')
print(df_bins.round(4))

In [ ]:
t1, t2 = np.percentile(qpp_scores, 33), np.percentile(qpp_scores, 66)
high = bert_scores[qpp_scores >= t2]
mid  = bert_scores[(qpp_scores >= t1) & (qpp_scores < t2)]
low  = bert_scores[qpp_scores < t1]

stat, p = kruskal(high, mid, low)
u_stat, u_p = mannwhitneyu(high, low, alternative='greater')
print(f'Kruskal-Wallis:   H={stat:.3f},  p={p:.4f}')
print(f'Mann-Whitney U:   U={u_stat:.1f}, p={u_p:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter: QPP vs BERTScore-F1
r, _ = pearsonr(qpp_scores, bert_scores)
axes[0].scatter(qpp_scores, bert_scores, alpha=0.4, s=15, c='#1E88E5')
m, b = np.polyfit(qpp_scores, bert_scores, 1)
xs   = np.linspace(qpp_scores.min(), qpp_scores.max(), 200)
axes[0].plot(xs, m * xs + b, 'r-', lw=2, label=f'r={r:.3f}')
axes[0].set_xlabel('Predicted QPP Score')
axes[0].set_ylabel('BERTScore-F1')
axes[0].set_title('QPP Score vs BERTScore-F1')
axes[0].legend()

# Bar: generation quality per QPP bin
bin_labels  = ['Low-QPP', 'Medium-QPP', 'High-QPP']
rouge_means = [bins['Low']['rouge_l'], bins['Medium']['rouge_l'], bins['High']['rouge_l']]
bert_means  = [bins['Low']['bert_f1'], bins['Medium']['bert_f1'], bins['High']['bert_f1']]
x = np.arange(len(bin_labels))
w = 0.35
axes[1].bar(x - w/2, rouge_means, w, label='ROUGE-L',      color='#43A047')
axes[1].bar(x + w/2, bert_means,  w, label='BERTScore-F1', color='#1E88E5')
axes[1].set_xticks(x)
axes[1].set_xticklabels(bin_labels)
axes[1].set_ylabel('Score')
axes[1].set_title('Generation Quality per QPP Bin')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/generation_correlation.png', dpi=150, bbox_inches='tight')
plt.show()